# 04 — Day-level 장소추정 캡션 보강 EDA
spec: docs/superpowers/specs/2026-06-04-day-geocaption-eda-design.md
입력: 03 핸드오프(vision_manifest.parquet + thumbs/ + pool_captions_03.parquet).
가설: 26b day-묶음 multi-image coarse 지명 → e2b 캡션 주입 → 03 지명약점(제주·일산 0.00) 보완.
전부 로컬 Ollama, 외부 전송 0(ADR-0001).

In [1]:
import json, time, random, re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import pillow_heif; pillow_heif.register_heif_opener()
import ollama

SEED = 42
random.seed(SEED); np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE = ROOT / "data" / "eda_cache"
MANIFEST = CACHE / "vision_manifest.parquet"
THUMB_DIR = CACHE / "thumbs"
POOL_CAP_03 = CACHE / "pool_captions_03.parquet"   # 03 평가풀 e2b 장면캡션

GEO_MODEL = "gemma4:26b"      # day-place 장소추정(사용자 지목, 지리 사전지식)
SCENE_MODEL = "gemma4:e2b"    # 03 장면캡션(재사용; 재생성 시 폴백용)
EMB_MODEL = "qwen3-embedding:8b"

In [2]:
installed = {x.model for x in ollama.list().models}
need = {GEO_MODEL, SCENE_MODEL, EMB_MODEL}
missing = need - installed
assert not missing, f"누락 모델: {missing} — ollama pull 필요"
print("Ollama OK:", sorted(need))

Ollama OK: ['gemma4:26b', 'gemma4:e2b', 'qwen3-embedding:8b']


In [3]:
m = pd.read_parquet(MANIFEST)
assert len(m) >= 1737, len(m)
assert m["thumb_path"].notna().all()

assert POOL_CAP_03.exists(), \
    "03 평가풀 캡션 없음 — notebooks/03_vision_caption_eda.ipynb의 풀 캡션 셀을 먼저 실행"
pool03 = pd.read_parquet(POOL_CAP_03)
pool03 = pool03[["thumb_path", "folder_top", "caption"]].drop_duplicates("thumb_path")
assert pool03["caption"].notna().all() and len(pool03) >= 400, len(pool03)
print(f"manifest {len(m)} · 03 평가풀 캡션 {len(pool03)}장 (e2b 장면캡션 재사용)")

manifest 3122 · 03 평가풀 캡션 400장 (e2b 장면캡션 재사용)


In [4]:
def montage(thumb_paths, cols=3, cell=384):
    """N장을 cols 그리드 1장으로 합성. multi-image FAIL 시 단일 입력 폴백."""
    paths = list(thumb_paths); n = len(paths); rows = (n + cols - 1) // cols
    canvas = Image.new("RGB", (cols*cell, rows*cell), (16, 16, 16))
    for i, tp in enumerate(paths):
        im = ImageOps.contain(Image.open(tp).convert("RGB"), (cell, cell))
        x = (i % cols)*cell + (cell - im.width)//2
        y = (i // cols)*cell + (cell - im.height)//2
        canvas.paste(im, (x, y))
    out = CACHE / "_montage_tmp.jpg"; canvas.save(out, quality=88)
    return str(out)

In [5]:
_p = []
for i, c in enumerate([(200,0,0),(0,200,0),(0,0,200)]):
    q = CACHE / f"_smk_{i}.jpg"; Image.new("RGB",(64,48),c).save(q); _p.append(str(q))
_mp = montage(_p, cols=3, cell=384)
_mi = Image.open(_mp)
assert _mi.size == (3*384, 1*384), _mi.size      # 3장→1행3열
assert montage(_p[:1]).__class__ is str          # 1장도 동작
print("montage assert OK", _mi.size)
for q in _p: Path(q).unlink()

montage assert OK (1152, 384)


In [6]:
# 서로 다른 폴더(=다른 장소) 1장씩 골라 묶음 입력 → 두 장면을 모두 인식하는지
two = (m.sort_values("folder_top").groupby("folder_top").head(1)
         .sample(2, random_state=SEED))
smoke_imgs = two["thumb_path"].tolist()
SMOKE_PROMPT = ("You are shown multiple photos at once. Briefly describe EACH photo "
    "separately as a numbered list (1., 2., ...). Do not merge them.")
r = ollama.chat(model=GEO_MODEL,
    messages=[{"role":"user","content":SMOKE_PROMPT,"images":smoke_imgs}],
    options={"seed":SEED})
resp = r["message"]["content"]
print("입력:", [Path(p).name for p in smoke_imgs]); print("---\n", resp)

입력: ['FileJPEG-1126.jpg.jpg', '180501_X-T3_DSCF3650.jpg.jpg']
---
 1. A photo of a young woman with long brown hair standing by a canal. She is wearing a white, long-sleeved shirt. The background features a row of brightly colored, multi-story buildings in shades of green, pink, white, and orange lining a canal. A small boat with a "SUZUKI" outboard motor is visible in the foreground.
2. A close-up photograph of lush green leaves and branches. The foliage is dense, with some leaves showing small brown spots. The background is a bright, nearly white, overexposed light.


In [7]:
# 휴리스틱: 번호목록 2개 이상 + 길이. 최종은 사람이 위 응답 보고 확정.
auto = bool(re.search(r"(?m)^\s*2[\.\)]", resp)) and len(resp) > 80
MULTI_IMAGE_OK = auto   # ← 위 응답이 두 장면을 따로 기술하면 True 유지, 아니면 False로 수정
print(f"휴리스틱 multi-image PASS={auto} → MULTI_IMAGE_OK={MULTI_IMAGE_OK}")
print("FAIL이면 montage fallback으로 진행(품질 동등, 해상도만 손실).")

휴리스틱 multi-image PASS=True → MULTI_IMAGE_OK=True
FAIL이면 montage fallback으로 진행(품질 동등, 해상도만 손실).


In [8]:
def day_groups(df):
    """folder_top → (exif_date 있으면) 날짜 세분. 날짜 결손은 폴더 1그룹."""
    d = df.copy()
    dt = pd.to_datetime(d.get("exif_date"), errors="coerce")
    d["day"] = dt.dt.strftime("%Y-%m-%d")
    d["group_key"] = np.where(d["day"].notna(),
                              d["folder_top"] + "|" + d["day"].fillna(""),
                              d["folder_top"])
    d["_sort"] = dt
    return d

def rep_sample(g, n=6):
    """그룹 내 시간순 균등 N장(하루 대표). 날짜 없으면 입력순. 결정적."""
    g = g.sort_values("_sort", na_position="last")
    if len(g) <= n: return g
    idx = np.linspace(0, len(g)-1, n).round().astype(int)
    return g.iloc[np.unique(idx)]

In [9]:
_df = pd.DataFrame({
    "folder_top": ["A"]*8 + ["B"]*2,
    "exif_date": ["2020-01-01"]*4 + ["2020-01-02"]*4 + [None, None],
    "thumb_path": [f"t{i}" for i in range(10)],
})
_g = day_groups(_df)
assert set(_g["group_key"]) == {"A|2020-01-01", "A|2020-01-02", "B"}, set(_g["group_key"])
_r = rep_sample(_g[_g.group_key=="A|2020-01-01"], n=6)
assert len(_r) == 4                          # 4장뿐이면 전량
_r2 = rep_sample(_g[_g.group_key=="A|2020-01-01"].iloc[[0]*10].assign(_sort=range(10)), n=6)
assert len(_r2) == 6                          # 10장→6장
print("day_groups / rep_sample assert OK")

day_groups / rep_sample assert OK


In [10]:
GEO_PROMPTS = {
  "aggressive": ("These photos are all from the SAME day or trip. Using only visual clues "
    "(terrain, vegetation, architecture, skyline, signage script, sky/light), infer the SINGLE "
    "most likely location at COARSE granularity (city or province + country). You MUST make your "
    "best guess even if uncertain. Reply with ONLY the place name, e.g. 'Jeju Island, South Korea'."),
  "conservative": ("These photos are all from the SAME day or trip. If the location is clearly "
    "identifiable from visual clues at COARSE granularity (city or province + country), reply with "
    "ONLY that place name; otherwise reply with exactly 'unknown'. Do not guess wildly."),
}

def estimate_place(thumb_paths, prompt, multi_ok):
    """multi_ok면 N장 직접 입력, 아니면 montage 1장. flag-skip."""
    try:
        imgs = list(thumb_paths) if multi_ok else [montage(thumb_paths)]
        r = ollama.chat(model=GEO_MODEL,
            messages=[{"role":"user","content":prompt,"images":imgs}], options={"seed":SEED})
        return r["message"]["content"].strip().splitlines()[0][:120]
    except Exception as e:
        return f"__ERR__:{e}"

In [11]:
PLACES = CACHE / "day_places_04.parquet"
poolm = day_groups(m[m.thumb_path.isin(pool03.thumb_path)])   # 03 풀이 덮는 사진만
if PLACES.exists():
    places = pd.read_parquet(PLACES)
else:
    recs, t0 = [], time.time()
    for gk, g in poolm.groupby("group_key"):
        reps = rep_sample(g, 6)
        rec = {"group_key": gk, "folder_top": g["folder_top"].iloc[0], "n": len(g)}
        for strat, ptext in GEO_PROMPTS.items():
            rec[strat] = estimate_place(reps["thumb_path"].tolist(), ptext, MULTI_IMAGE_OK)
        recs.append(rec)
    places = pd.DataFrame(recs); places.to_parquet(PLACES)
    errs = places[["aggressive","conservative"]].apply(lambda s: s.str.startswith("__ERR__")).sum().sum()
    print(f"{len(places)} 그룹 추정 · {time.time()-t0:.0f}s · 에러 {int(errs)}")
print(places[["group_key","n","aggressive","conservative"]].head(20).to_string())

                                group_key   n                                                                                                                aggressive              conservative
0   170415_NIKON D610_DSC_8784_white2.png   1                                                                                                               South Korea                   unknown
1                180410_X-T3_DSCF3222.jpg   1                                                                                                        Seoul, South Korea                   unknown
2                180410_X-T3_DSCF3228.jpg   1                                                                                                            Moscow, Russia                   unknown
3                         181229_30_busan  25                                                                                                  Jeju Island, South Korea  Jeju Island, South Korea
4                        19021

/tmp/claude-501/ipykernel_54254/1260138443.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(d.get("exif_date"), errors="coerce")


In [12]:
def combine(scene_cap, place):
    """e2b 장면캡션 + Location 줄. unknown/에러/빈값은 미부착(=기준선과 동일)."""
    p = (place or "").strip()
    if not p or p.lower() == "unknown" or p.startswith("__ERR__"):
        return scene_cap
    return f"{scene_cap}\nLocation: {p}"

def embed(text):   # 03과 동일 시그니처
    return np.asarray(ollama.embed(model=EMB_MODEL, input=text).embeddings[0], dtype=np.float32)


In [13]:
assert combine("a cat", "Jeju Island, South Korea") == "a cat\nLocation: Jeju Island, South Korea"
assert combine("a cat", "unknown") == "a cat"
assert combine("a cat", "__ERR__:x") == "a cat"
assert combine("a cat", "") == "a cat"
print("combine assert OK")


combine assert OK


In [14]:
poolj = (poolm.merge(places[["group_key","aggressive","conservative"]], on="group_key", how="left")
              .merge(pool03[["thumb_path","caption"]], on="thumb_path", how="left"))
assert poolj["caption"].notna().all(), "03 캡션 매칭 실패 thumb 존재"
poolj["text_base"] = poolj["caption"]                                              # e2b 단독(기준선)
poolj["text_aggr"] = [combine(c,p) for c,p in zip(poolj.caption, poolj.aggressive)]  # +place 공격
poolj["text_cons"] = [combine(c,p) for c,p in zip(poolj.caption, poolj.conservative)]# +place 보수
print("place 부착률 · 공격:", (poolj.text_aggr!=poolj.text_base).mean().round(3),
      "· 보수:", (poolj.text_cons!=poolj.text_base).mean().round(3))


place 부착률 · 공격: 1.0 · 보수: 0.8


In [15]:
EMB = CACHE / "pool_embeddings_04.parquet"
if EMB.exists():
    emb_df = pd.read_parquet(EMB)
else:
    t0 = time.time()
    for col in ["text_base","text_aggr","text_cons"]:
        poolj[col+"_emb"] = [embed(t).tolist() for t in poolj[col]]
    emb_df = poolj; emb_df.to_parquet(EMB)
    print(f"임베딩 {len(emb_df)}×3 · {time.time()-t0:.0f}s")
assert all(c in emb_df for c in ["text_base_emb","text_aggr_emb","text_cons_emb"])


In [16]:
# Task 5 Step 1: 한국어 질의셋 (03 노트북과 동일)
import unicodedata

QUERIES = {
  "결혼식 사진": "wedding",
  "아이슬란드 여행": "2022_아이슬란드",
  "이탈리아 여행": "2019_이탈리아",
  "제주도에서 찍은 사진": "200620_23_제주",
  "방콕 여행": "2018방콕",
  "부산에서 찍은 사진": "181229_30_busan",
  "단양 여행": "201904_단양",
  "몽골 여행": "2018몽골",
  "개심사 절 사진": "20190505_개심사",
  "일산 호수공원": "20-04-05_일산호수공원",
}
GEO_QUERIES = ["제주도에서 찍은 사진", "일산 호수공원"]   # 03에서 recall@10 0.00
# macOS NFD 주의: emb_df·pool03 folder_top은 NFD일 수 있으므로 NFC 정규화 후 비교
nfc = lambda s: unicodedata.normalize('NFC', str(s))
folders = {nfc(v) for v in pool03['folder_top']} | {nfc(v) for v in m['folder_top']}
miss = [v for v in QUERIES.values() if nfc(v) not in folders]
assert not miss, f"정답 폴더 불일치(NFC 주의): {miss}"
print(len(QUERIES), "질의 · 지명질의:", GEO_QUERIES)


10 질의 · 지명질의: ['제주도에서 찍은 사진', '일산 호수공원']


In [17]:
# Task 5 Step 2: recall@k 순수 함수 + assert (03과 동일)
def recall_mrr(q_emb, pool_embs, pool_labels, gold, ks=(5,10)):
    sims = pool_embs @ q_emb / (np.linalg.norm(pool_embs,axis=1)*np.linalg.norm(q_emb)+1e-9)
    order = np.argsort(-sims)
    hits = [i for i,idx in enumerate(order) if pool_labels[idx]==gold]
    rr = 1.0/(hits[0]+1) if hits else 0.0
    return {f"recall@{k}": float(any(h < k for h in hits)) for k in ks} | {"mrr": rr}

_pe = np.array([[1,0],[0,1],[0.9,0.1]], dtype=np.float32)
_r = recall_mrr(np.array([1,0],dtype=np.float32), _pe, ["g","x","g"], "g")
assert _r["recall@5"]==1.0 and abs(_r["mrr"]-1.0)<1e-6, _r
print("recall_mrr assert OK")


recall_mrr assert OK


In [18]:
# Task 5 Step 3: 비교군 3 × 질의 recall
# macOS NFD 주의: emb_df folder_top은 NFD → NFC 정규화 후 비교
ARMS = {"e2b_base":"text_base_emb", "+place_aggr":"text_aggr_emb", "+place_cons":"text_cons_emb"}
rows = []
for arm, col in ARMS.items():
    P = np.array(emb_df[col].tolist(), dtype=np.float32)
    labels = [unicodedata.normalize('NFC', str(v)) for v in emb_df['folder_top']]
    for q, gold in QUERIES.items():
        gold_nfc = unicodedata.normalize('NFC', gold)
        rows.append({"arm":arm, "query":q, **recall_mrr(embed(q), P, labels, gold_nfc)})
res = pd.DataFrame(rows)
print("== 비교군 평균 =="); print(res.groupby("arm")[["recall@5","recall@10","mrr"]].mean().round(3))
print("\n== 질의별 recall@10 ==")
print(res.pivot_table(index="query", columns="arm", values="recall@10"))
print("\n== 지명질의(제주·일산) recall@10 ==")
print(res[res["query"].isin(GEO_QUERIES)].pivot_table(index="query", columns="arm", values="recall@10"))


== 비교군 평균 ==
             recall@5  recall@10    mrr
arm                                    
+place_aggr       0.5        0.6  0.468
+place_cons       0.4        0.5  0.384
e2b_base          0.3        0.6  0.285

== 질의별 recall@10 ==
arm          +place_aggr  +place_cons  e2b_base
query                                          
개심사 절 사진             1.0          0.0       1.0
결혼식 사진               1.0          1.0       1.0
단양 여행                1.0          1.0       1.0
몽골 여행                0.0          0.0       0.0
방콕 여행                1.0          1.0       1.0
부산에서 찍은 사진           0.0          0.0       0.0
아이슬란드 여행             1.0          1.0       1.0
이탈리아 여행              1.0          1.0       1.0
일산 호수공원              0.0          0.0       0.0
제주도에서 찍은 사진          0.0          0.0       0.0

== 지명질의(제주·일산) recall@10 ==
arm          +place_aggr  +place_cons  e2b_base
query                                          
일산 호수공원              0.0          0.0       0.0
제주도에서 찍은 사진      

In [19]:
# Task 5 Step 4: day-place 정확도·환각율 (alias 규칙 + 정성)
PLACE_ALIAS = {
  "jeju":"제주", "iceland":"아이슬란드", "italy":"이탈리아", "dolomites":"이탈리아",
  "bangkok":"방콕", "busan":"busan", "danyang":"단양", "mongolia":"몽골",
  "ilsan":"일산", "seoul":"서울", "gaesimsa":"개심사", "gangneung":"강릉",
}
def place_hit(place, folder_top):
    f = unicodedata.normalize("NFC", str(folder_top)).lower(); pl = str(place).lower()
    for en, ko in PLACE_ALIAS.items():
        if en in pl and (en in f or ko in f): return True
    return False

ev = places.copy()
for strat in ["aggressive","conservative"]:
    named = ~ev[strat].str.lower().isin(["unknown"]) & ~ev[strat].str.startswith("__ERR__")
    hit = ev.apply(lambda r: place_hit(r[strat], r["folder_top"]), axis=1)
    n_named = int(named.sum())
    print(f"[{strat}] 추정시도 {n_named}/{len(ev)} · 적중(alias) {int((named&hit).sum())} "
          f"· 미적중(환각후보) {int((named&~hit).sum())} · unknown {int((~named).sum())}")
print("\n== 환각 후보(추정했으나 alias 불일치) — 정성검토 ==")
print(ev[named & ~ev.apply(lambda r: place_hit(r['aggressive'], r['folder_top']), axis=1)]
        [["group_key","folder_top","aggressive"]].head(20).to_string())


[aggressive] 추정시도 29/29 · 적중(alias) 7 · 미적중(환각후보) 22 · unknown 0
[conservative] 추정시도 14/29 · 적중(alias) 6 · 미적중(환각후보) 8 · unknown 15

== 환각 후보(추정했으나 alias 불일치) — 정성검토 ==
             group_key          folder_top                aggressive
3      181229_30_busan     181229_30_busan  Jeju Island, South Korea
9           2018몽골          2018몽골             Colorado, USA
11       201904_단양       201904_단양               South Korea
21             wedding             wedding  Jeju Island, South Korea
22                기타                기타                     Italy
23  마음의 빚 우토로  마음의 빚 우토로              Kyoto, Japan
27               은지               은지        Busan, South Korea
28               하율               하율          Dolomites, Italy
